In [1]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

C:\Users\valentin\.conda\envs\mirea-torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    # Настройки для более детерминированного поведения на GPU (если доступно).
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

    # Включаем детерминированные алгоритмы (PyTorch >=1.8). При необходимости окружение CUDA
    # может требовать настройки переменных CUBLAS_WORKSPACE_CONFIG или CPU реализаций.
    # try:
    #     torch.use_deterministic_algorithms(True)
    # except Exception:
    #     try:
    #         torch.set_deterministic(True)
    #     except Exception:
    #         pass

    # Сделаем генератор для DataLoader доступным глобально, чтобы все загрузчики были детерминированы
    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [5]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()
api.dataset_download_files("alexandersemiletov/toxic-russian-comments", path=str(data_dir), unzip=True, force=True)

Dataset URL: https://www.kaggle.com/datasets/alexandersemiletov/toxic-russian-comments


In [6]:
data_list = []
with open("../data/dataset.txt", encoding="utf-8") as file:
    for line in file:
        labels = line.split()[0]
        text = line[len(labels) + 1:].strip()
        labels = labels.split(",")
        mask = [
            1 if "__label__NORMAL" in labels else 0,
            1 if "__label__INSULT" in labels else 0,
            1 if "__label__THREAT" in labels else 0,
            1 if "__label__OBSCENITY" in labels else 0,
        ]
        data_list.append((text, *mask))

In [7]:
df = pd.DataFrame(data_list, columns=["text", "normal", "insult", "threat", "obscenity"])

In [8]:
df.head()

,text,normal,insult,threat,obscenity
0,скотина! что сказать,0,1,0,0
1,я сегодня проезжала по рабочей и между домами ...,1,0,0,0
2,очередной лохотрон. зачем придумывать очередно...,1,0,0,0
3,"ретро дежавю ... сложно понять чужое сердце , ...",1,0,0,0
4,а когда мы статус агрогородка получили?,1,0,0,0


In [9]:

initial_rows = len(df)

label_cols = ["normal", "insult", "threat", "obscenity"]

df["labels_tuple"] = df[label_cols].apply(tuple, axis=1)
labels_nunique = df.groupby("text")["labels_tuple"].nunique()
conflict_texts = labels_nunique[labels_nunique > 1].index.tolist()
conflict_set = set(conflict_texts)

removed_conflicting_df = df[df["text"].isin(conflict_set)].copy()

mask_nonconflict = ~df["text"].isin(conflict_set)
df_nonconflict = df[mask_nonconflict].copy()

exact_dup_mask = df_nonconflict.duplicated(subset=["text"] + label_cols, keep="first")
removed_exact_df = df_nonconflict[exact_dup_mask].copy()

df_clean = df_nonconflict.drop_duplicates(subset=["text"] + label_cols, keep="first").reset_index(drop=True)
if "labels_tuple" in df_clean.columns:
    df_clean = df_clean.drop(columns=["labels_tuple"])

final_rows = len(df_clean)
num_removed_exact = len(removed_exact_df)
num_removed_conflicting = len(removed_conflicting_df)



print("initial_rows -", initial_rows)
print("removed exact duplicate rows -", num_removed_exact)
print("removed conflicting rows (all variants) -", num_removed_conflicting)
print("final_rows -", final_rows)
print("sum check:", initial_rows, "==", final_rows + num_removed_exact + num_removed_conflicting)


initial_rows - 248290
removed exact duplicate rows - 6
removed conflicting rows (all variants) - 2
final_rows - 248282
sum check: 248290 == 248290


In [10]:
df = df_clean
df.shape


(248282, 5)

In [11]:
# Берём 1/10 датасета для ускорения обучения (10-я часть)
label_cols = ["normal", "insult", "threat", "obscenity"]
X = df["text"].astype(str)
y = df[label_cols]

sample_size = len(df) // 10
if MultilabelStratifiedShuffleSplit is not None:
    msss_sample = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=1.0 - (sample_size / len(df)), random_state=42)
    for sample_idx, _ in msss_sample.split(X, y):
        sample_idx = X.index[sample_idx]
        df = df.loc[sample_idx].reset_index(drop=True)
else:
    strat = y.sum(axis=1)
    sample_indices = train_test_split(df.index, train_size=sample_size / len(df), random_state=42, stratify=strat)[0]
    df = df.loc[sample_indices].reset_index(drop=True)

print(f"Sampled dataset size: {len(df)} (1/10 of original)")


Sampled dataset size: 24826 (1/10 of original)


In [12]:
# Split dataset to train/val/test = 70/15/15 with stratification (multilabel-aware if available)
label_cols = ["normal", "insult", "threat", "obscenity"]
X = df["text"].astype(str)
y = df[label_cols]

try:
    if MultilabelStratifiedShuffleSplit is not None:
        msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
        for train_idx, temp_idx in msss.split(X, y):
            X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
            y_train, y_temp = y.iloc[train_idx], y.iloc[temp_idx]
        # split temp into val/test equally
        msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
        for val_idx_rel, test_idx_rel in msss2.split(X_temp, y_temp):
            val_idx = X_temp.index[val_idx_rel]
            test_idx = X_temp.index[test_idx_rel]
            X_val, X_test = X.loc[val_idx], X.loc[test_idx]
            y_val, y_test = y.loc[val_idx], y.loc[test_idx]
    else:
        # fallback: stratify by label sum
        strat = y.sum(axis=1)
        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=strat)
        strat_temp = y_temp.sum(axis=1)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=strat_temp)
except Exception:
    # final fallback: random split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("sizes: train={}, val={}, test={}".format(len(X_train), len(X_val), len(X_test)))


split_dir = data_dir / "splits"
split_dir.mkdir(exist_ok=True)
train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
val_df = pd.concat([X_val.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)



sizes: train=17379, val=3730, test=3717


In [13]:
# Показать долю меток и комбинированных меток в train/val/test
label_cols = ["normal", "insult", "threat", "obscenity"]
# Доли меток и комбинированных меток в одном датафрейме для train/val/test
def shares_with_combos(df_split, top_k=20):

    # Комбинированные метки
    combos = df_split.apply(lambda r:
        ", ".join([c for c in label_cols if r[c] == 1]) if r[label_cols].sum() > 0 else "none",
        axis=1)
    counts_combo = combos.value_counts()
    shares_combo = (counts_combo / len(df_split) * 100).round(3)
    df_combos = pd.DataFrame({'count': counts_combo.astype(int), 'share_%': shares_combo})
    df_combos.index.name = 'label_or_combo'
    df_combos = df_combos.head(top_k)
    # Объединяем
    df_all = pd.concat([df_combos])
    return df_all

print('Train:', len(train_df))
display(shares_with_combos(train_df, top_k=20))
print('\nValidation:', len(val_df))
display(shares_with_combos(val_df, top_k=20))
print('\nTest:', len(test_df))
display(shares_with_combos(test_df, top_k=20))



Train: 17379


,count,share_%
label_or_combo,,
normal,14257,82.036
insult,2003,11.525
"insult, threat",436,2.509
threat,385,2.215
obscenity,153,0.880
"insult, obscenity",125,0.719
"insult, threat, obscenity",14,0.081
"threat, obscenity",6,0.035



Validation: 3730


,count,share_%
label_or_combo,,
normal,3055,81.903
insult,436,11.689
"insult, threat",89,2.386
threat,86,2.306
obscenity,37,0.992
"insult, obscenity",22,0.590
"insult, threat, obscenity",5,0.134



Test: 3717


,count,share_%
label_or_combo,,
normal,3056,82.217
insult,419,11.273
"insult, threat",106,2.852
threat,72,1.937
obscenity,36,0.969
"insult, obscenity",25,0.673
"insult, threat, obscenity",3,0.081


In [14]:
# Бейзлайн: DummyClassifier (most_frequent) для multilabel
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics_multilabel(y_true, y_pred, y_proba=None):
    acc = float(accuracy_score(y_true, y_pred))
    prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    if y_proba is not None:
        try:
            roc = float(roc_auc_score(y_true, y_proba, average="macro"))
        except Exception:
            roc = None
    else:
        roc = None
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": roc}

# обучаем на train, валидируем на val
X_train_text = train_df["text"].values.reshape(-1, 1)
X_val_text = val_df["text"].values.reshape(-1, 1)
y_train = train_df[label_cols].values
y_val = val_df[label_cols].values

dummy = OneVsRestClassifier(DummyClassifier(strategy="most_frequent"), n_jobs=-1)
dummy.fit(X_train_text, y_train)
y_pred = dummy.predict(X_val_text)
y_proba = None
if hasattr(dummy, "predict_proba"):
    try:
        y_proba = dummy.predict_proba(X_val_text)
    except Exception:
        y_proba = None

metrics = compute_metrics_multilabel(y_val, y_pred, y_proba)
metrics["model"] = "Dummy-most_frequent"
results_df = pd.DataFrame([metrics])
print(results_df)


   accuracy  precision  recall        f1  roc_auc                model
0  0.819035   0.204759    0.25  0.225129      0.5  Dummy-most_frequent


In [15]:
# Бейзлайн: SVM (LinearSVC) с TF-IDF, кросс-валидация и подбор гиперпараметров
# ВАЖНО: Без утечки данных — GridSearchCV на чистом train
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)),
    ("svm", OneVsRestClassifier(LinearSVC(random_state=42, dual=False, max_iter=2000), n_jobs=-1))
])

svm_params = {
    "svm__estimator__C": [0.1, 1, 10],
}

svm_grid = GridSearchCV(svm_pipeline, svm_params, cv=3, scoring="roc_auc", n_jobs=-1, verbose=1)
svm_grid.fit(X_train, y_train)

print("SVM best params:", svm_grid.best_params_)
print("SVM best CV score:", svm_grid.best_score_)

# Используем best_estimator_ (это уже Pipeline, обученный на всем train)
best_C = svm_grid.best_params_['svm__estimator__C']
tfidf_fitted = svm_grid.best_estimator_.named_steps['tfidf']
X_val_tfidf = tfidf_fitted.transform(X_val)

svm_best = svm_grid.best_estimator_.named_steps['svm']
y_pred_svm = svm_best.predict(X_val_tfidf)
y_proba_svm = svm_best.predict_proba(X_val_tfidf) if hasattr(svm_best, "predict_proba") else None

metrics_svm = compute_metrics_multilabel(y_val, y_pred_svm, y_proba_svm)
metrics_svm["model"] = "SVM-LinearSVC-TFIDF-GridSearch"
results_df = pd.concat([results_df, pd.DataFrame([metrics_svm])], ignore_index=True)
print(results_df)


Fitting 3 folds for each of 3 candidates, totalling 9 fits
SVM best params: {'svm__estimator__C': 0.1}
SVM best CV score: 0.9210928714843059
   accuracy  precision    recall        f1  roc_auc  \
0  0.819035   0.204759  0.250000  0.225129      0.5   
1  0.870509   0.881245  0.450704  0.535728      NaN   

                            model  
0             Dummy-most_frequent  
1  SVM-LinearSVC-TFIDF-GridSearch  


In [16]:
# Бейзлайн: Decision Tree с TF-IDF, кросс-валидация и подбор гиперпараметров
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)),
    ("dt", OneVsRestClassifier(DecisionTreeClassifier(random_state=42), n_jobs=-1))
])

dt_params = {
    "dt__estimator__max_depth": [10, 15, 20],
    "dt__estimator__min_samples_leaf": [2, 5],
}

dt_grid = GridSearchCV(dt_pipeline, dt_params, cv=3, scoring="roc_auc", n_jobs=-1, verbose=1)
dt_grid.fit(X_train, y_train)

print("DecisionTree best params:", dt_grid.best_params_)
print("DecisionTree best CV score:", dt_grid.best_score_)

y_pred_dt = dt_grid.predict(X_val)
y_proba_dt = dt_grid.predict_proba(X_val) if hasattr(dt_grid, "predict_proba") else None

metrics_dt = compute_metrics_multilabel(y_val, y_pred_dt, y_proba_dt)
metrics_dt["model"] = "DecisionTree-TFIDF-GridSearch"
results_df = pd.concat([results_df, pd.DataFrame([metrics_dt])], ignore_index=True)
print(results_df)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
DecisionTree best params: {'dt__estimator__max_depth': 20, 'dt__estimator__min_samples_leaf': 5}
DecisionTree best CV score: 0.7103758327370334
   accuracy  precision    recall        f1   roc_auc  \
0  0.819035   0.204759  0.250000  0.225129  0.500000   
1  0.870509   0.881245  0.450704  0.535728       NaN   
2  0.845040   0.815973  0.525018  0.611382  0.698503   

                            model  
0             Dummy-most_frequent  
1  SVM-LinearSVC-TFIDF-GridSearch  
2   DecisionTree-TFIDF-GridSearch  


In [17]:
# Бейзлайн: Gradient Boosting (HistGradientBoostingClassifier) с TF-IDF, кросс-валидация и подбор гиперпараметров
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer

def to_dense(X):
    if hasattr(X, "toarray"):
        return X.toarray()
    return X

gb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=1000, min_df=2)),
    ("to_dense", FunctionTransformer(to_dense, validate=False)),
    ("gb", OneVsRestClassifier(HistGradientBoostingClassifier(random_state=42, max_iter=50), n_jobs=-1))
])

gb_params = {
    "gb__estimator__learning_rate": [0.1],
    "gb__estimator__max_depth": [3, 5],
}

gb_grid = GridSearchCV(gb_pipeline, gb_params, cv=3, scoring="roc_auc", n_jobs=-1, verbose=1)
gb_grid.fit(X_train, y_train)

print("GradientBoosting best params:", gb_grid.best_params_)
print("GradientBoosting best CV score:", gb_grid.best_score_)

y_pred_gb = gb_grid.predict(X_val)
y_proba_gb = gb_grid.predict_proba(X_val) if hasattr(gb_grid, "predict_proba") else None

metrics_gb = compute_metrics_multilabel(y_val, y_pred_gb, y_proba_gb)
metrics_gb["model"] = "GradientBoosting-TFIDF-GridSearch"
results_df = pd.concat([results_df, pd.DataFrame([metrics_gb])], ignore_index=True)
print(results_df)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
GradientBoosting best params: {'gb__estimator__learning_rate': 0.1, 'gb__estimator__max_depth': 5}
GradientBoosting best CV score: 0.7836162888999264
   accuracy  precision    recall        f1   roc_auc  \
0  0.819035   0.204759  0.250000  0.225129  0.500000   
1  0.870509   0.881245  0.450704  0.535728       NaN   
2  0.845040   0.815973  0.525018  0.611382  0.698503   
3  0.852547   0.868828  0.410518  0.486117  0.806013   

                               model  
0                Dummy-most_frequent  
1     SVM-LinearSVC-TFIDF-GridSearch  
2      DecisionTree-TFIDF-GridSearch  
3  GradientBoosting-TFIDF-GridSearch  


In [ ]:
# GRU для multilabel классификации текстов
from torch.nn import Embedding
from torch.utils.data import TensorDataset

class GRUTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 100, hidden_size: int = 64,
                 num_layers: int = 1, dropout: float = 0.2, num_classes: int = 4):
        super().__init__()
        self.embedding = Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.gru(emb)
        last_hidden = out[:, -1, :]
        x = self.dropout(last_hidden)
        logits = self.head(x)
        return logits

# Создаём словарь слов из train текстов
word2idx = {"<PAD>": 0}
idx = 1
for text in X_train:
    for word in text.lower().split():
        if word not in word2idx:
            word2idx[word] = idx
            idx += 1

print(f"Vocab size: {len(word2idx)}")

def texts_to_sequences(texts, word2idx, max_len=100):
    sequences = []
    for text in texts:
        seq = [word2idx.get(word, 0) for word in text.lower().split()][:max_len]
        seq += [0] * (max_len - len(seq))
        sequences.append(seq)
    return np.array(sequences)

max_len = 100
X_train_seq = texts_to_sequences(X_train.values, word2idx, max_len)
X_val_seq = texts_to_sequences(X_val.values, word2idx, max_len)

X_train_tensor = torch.LongTensor(X_train_seq).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
X_val_tensor = torch.LongTensor(X_val_seq).to(device)
y_val_tensor = torch.FloatTensor(y_val).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model_gru = GRUTextClassifier(vocab_size=len(word2idx), embed_dim=100, hidden_size=64,
                               num_layers=2, dropout=0.3, num_classes=4).to(device)

optimizer = AdamW(model_gru.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

print("Training GRU...")
epochs = 10
for epoch in range(epochs):
    model_gru.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for X_batch, y_batch in pbar:
        optimizer.zero_grad()
        logits = model_gru(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({"loss": total_loss / (pbar.n + 1)})

    model_gru.eval()
    with torch.no_grad():
        val_logits = model_gru(X_val_tensor)
        val_loss = loss_fn(val_logits, y_val_tensor)
        val_preds = (torch.sigmoid(val_logits) > 0.5).float()

    print(f"Epoch {epoch+1}/{epochs} | train_loss: {total_loss/len(train_loader):.4f} | val_loss: {val_loss.item():.4f}\n")

print("\nGRU training complete!")

with torch.no_grad():
    y_pred_gru = (torch.sigmoid(model_gru(X_val_tensor)) > 0.5).float().cpu().numpy()
    y_proba_gru = torch.sigmoid(model_gru(X_val_tensor)).cpu().numpy()

metrics_gru = compute_metrics_multilabel(y_val, y_pred_gru, y_proba_gru)
metrics_gru["model"] = "GRU-Text-Classifier"
results_df = pd.concat([results_df, pd.DataFrame([metrics_gru])], ignore_index=True)
print(results_df)

In [ ]:
# Заменённая ячейка: LSTM — диагностика и исправленное обучение
from torch.nn import Embedding
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 100, hidden_size: int = 128,
                 num_layers: int = 2, dropout: float = 0.3, num_classes: int = 4):
        super().__init__()
        self.embedding = Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (hn, cn) = self.lstm(emb)
        last_hidden = out[:, -1, :]
        x = self.dropout(last_hidden)
        logits = self.head(x)
        return logits

max_len = 100

def texts_to_sequences(texts, word2idx, max_len=100):
    sequences = []
    for text in texts:
        seq = [word2idx.get(word, 0) for word in text.lower().split()][:max_len]
        seq += [0] * (max_len - len(seq))
        sequences.append(seq)
    return np.array(sequences)

if 'word2idx' not in globals():
    word2idx = {"<PAD>": 0}
    idx = 1
    for text in X_train:
        for word in text.lower().split():
            if word not in word2idx:
                word2idx[word] = idx
                idx += 1

X_train_seq = texts_to_sequences(X_train.values, word2idx, max_len)
X_val_seq = texts_to_sequences(X_val.values, word2idx, max_len)

X_train_tensor = torch.LongTensor(X_train_seq).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
X_val_tensor = torch.LongTensor(X_val_seq).to(device)
y_val_tensor = torch.FloatTensor(y_val).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model_lstm = LSTMTextClassifier(vocab_size=len(word2idx), embed_dim=100, hidden_size=128,
                                num_layers=4, dropout=0.3, num_classes=y_train.shape[1]).to(device)

optimizer = AdamW(model_lstm.parameters(), lr=5e-4)

# Use unweighted loss for LSTM (remove pos_weight) to avoid biasing predictions by class weight
loss_fn = nn.BCEWithLogitsLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

print('Label distribution (train):', y_train.sum(axis=0))
# print('pos_weight used:', pos_weight.cpu().numpy())

model_lstm.train()
with torch.no_grad():
    sample_logits = model_lstm(X_train_tensor[:min(16, X_train_tensor.size(0))])
    sample_probs = torch.sigmoid(sample_logits)
    print('Initial sample logits mean:', sample_logits.mean().item(), 'probs mean:', sample_probs.mean().item())

best_val_loss = float('inf')
early_patience = 5
no_improve = 0
epochs = 20

for epoch in range(epochs):
    model_lstm.train()
    total_loss = 0.0
    processed = 0
    pbar = tqdm(total=len(train_loader.dataset), desc=f"LSTM Epoch {epoch+1}/{epochs}", unit="sample")
    grad_norms = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        logits = model_lstm(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=5.0)
        optimizer.step()
        step_batch = X_batch.size(0)
        processed += step_batch
        total_loss += loss.item() * step_batch
        pbar.update(step_batch)
        pbar.set_postfix({'loss': f'{total_loss/processed:.4f}'})
    pbar.close()

    model_lstm.eval()
    with torch.no_grad():
        val_logits = model_lstm(X_val_tensor)
        val_loss = loss_fn(val_logits, y_val_tensor).item()
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_preds = (val_probs > 0.5).astype(int)

    scheduler.step(val_loss)

    if val_loss + 1e-6 < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = model_lstm.state_dict()
    else:
        no_improve += 1

    # compute and log a few diagnostics
    with torch.no_grad():
        logits_sample = model_lstm(X_train_tensor[:min(16, X_train_tensor.size(0))])
        probs_sample = torch.sigmoid(logits_sample).cpu().numpy()
    print(f"Epoch {epoch+1}/{epochs} train_loss={total_loss/len(train_loader.dataset):.4f} val_loss={val_loss:.4f} processed={processed}")
    print('Sample probs mean:', probs_sample.mean(), 'sample preds sum:', probs_sample.sum())

    if no_improve >= early_patience:
        print('Early stopping')
        break

model_lstm.load_state_dict(best_state)

with torch.no_grad():
    logits = model_lstm(X_val_tensor)
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs > 0.5).astype(int)/

metrics_lstm = compute_metrics_multilabel(y_val, preds, probs)
metrics_lstm['model'] = 'LSTM-Text-Classifier'
results_df = pd.concat([results_df, pd.DataFrame([metrics_lstm])], ignore_index=True)
print(results_df)